In [ ]:
# 1. Instalar as bibliotecas necessárias no Colab
!pip install -q groq pandas tqdm pydantic openpyxl

import os
import json
import time
import pandas as pd
from tqdm import tqdm
from pydantic import BaseModel
from typing import Literal
from groq import Groq

# ==========================================
# 2. SUA CHAVE DA GROQ JÁ CONFIGURADA
# ==========================================
# Credential removed for public archive. Use GROQ_API_KEY from environment or Colab Secrets.
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

# Seleciona o Qwen da sua lista ativa na Groq
GROQ_MODEL = "qwen/qwen3.8-27b"
print(f"✅ Usando o modelo principal da Groq: {GROQ_MODEL}")

# ==========================================
# 3. SCHEMA DE SAÍDA E SYSTEM PROMPT
# ==========================================
class AnotacaoResponsividade(BaseModel):
    objetivo_central: str
    mesmo_tema: bool
    aborda_ponto_especifico: str
    rotulo: Literal["RESPONDE", "PARCIAL", "ESQUIVA"]

SYSTEM_PROMPT = """Você é um anotador especialista em Ciência Política e Linguística Computacional analisando entrevistas políticas em português brasileiro do programa Roda Viva.
Sua tarefa é classificar o nível de responsividade da RESPOSTA do político em relação à PERGUNTA do jornalista em uma de três categorias: RESPONDE, PARCIAL ou ESQUIVA.

### DEFINIÇÕES DAS CATEGORIAS
- RESPONDE: O político aborda DIRETAMENTE o ponto central da pergunta. A resposta satisfaz o objetivo comunicativo principal do jornalista, mesmo que não cubra todos os aspectos secundários.
- PARCIAL: O político aborda o TEMA da pergunta, mas evita aspectos-chave, introduz digressões significativas ou dá uma resposta incompleta que deixa o ponto central sem resposta.
- ESQUIVA: O político redireciona para outro assunto, responde a uma pergunta diferente da feita, ou usa estratégias retóricas (atacar o entrevistador, mudar de tema) para evitar o ponto central.

### ÁRVORE DE DECISÃO (SIGA ESTES 3 PASSOS)
PASSO 1: Identifique o OBJETIVO CENTRAL da pergunta. Pergunte: "O que o jornalista quer saber de fato?" Ignore a conversa de abertura e contextualizações longas. Foque no que é pedido.
PASSO 2: O político fala sobre o MESMO TEMA da pergunta?
-> NÃO: classifique como ESQUIVA (o político mudou de assunto completamente).
-> SIM: vá para o Passo 3.
PASSO 3: O político aborda o PONTO ESPECÍFICO que foi perguntado?
-> SIM: classifique como RESPONDE (ele respondeu diretamente o que foi perguntado).
-> NÃO / PARCIALMENTE: classifique como PARCIAL (ele falou do tema, mas evitou o ponto central).

### REGRAS PRAGMÁTICAS CRÍTICAS (ATENÇÃO)
1. Extensão != qualidade: Uma resposta longa NÃO é automaticamente RESPONDE. Um político pode falar por vários minutos sobre o tema sem responder à pergunta específica. Foque no cumprimento do objetivo central, não no tamanho do texto. Da mesma forma, uma resposta curta e direta que atende ao que foi perguntado é RESPONDE.
2. Pergunta com duas ou mais partes: Foque na parte PRINCIPAL (geralmente a última, mais específica). Se o político responde apenas à parte secundária, classifique como PARCIAL.
3. Político elogia o entrevistador e desvia: Se o desvio é claro e o ponto central não é abordado -> ESQUIVA.
4. Resposta vaga ou genérica sobre o tema: Se não satisfaz o que foi perguntado especificamente -> PARCIAL, não RESPONDE.
5. Recusa explícita ou desconhecimento: Se o político diz diretamente "não sei" ou "não posso responder", isso É uma resposta direta -> RESPONDE.

### EXEMPLOS DE REFERÊNCIA
Exemplo 1:
PERGUNTA: "Governador, o senhor é candidato para ganhar ou para atrapalhar?"
RESPOSTA: "Olha, eu não vou ser candidato para atrapalhar, eu sou candidato para ganhar. Não sou candidato para atrapalhar ninguém. Agora, veja: o PT fez o governo mais corrupto da história do Brasil..."
SAÍDA: {"objetivo_central": "Saber se o político é candidato para ganhar ou para atrapalhar.", "mesmo_tema": true, "aborda_ponto_especifico": "Sim, escolhe diretamente uma das opções ('sou candidato para ganhar') antes de acrescentar contexto.", "rotulo": "RESPONDE"}

Exemplo 2:
PERGUNTA: "É verdade que o senhor outro dia estava conversando com o ex-presidente Fernando Henrique sobre uma possível candidatura?"
RESPOSTA: "Mas o Fernando Henrique é um grande frasista. Eu tenho um profundo respeito pelo presidente Fernando Henrique. Agora, o que eu acho é que o Brasil precisa de uma liderança forte e unida..."
SAÍDA: {"objetivo_central": "Confirmar ou negar um encontro específico com FHC sobre candidatura.", "mesmo_tema": false, "aborda_ponto_especifico": "Não confirma nem nega o encontro; redireciona para elogios e declarações gerais.", "rotulo": "ESQUIVA"}

Exemplo 3:
PERGUNTA: "Governador, esse seu choque de gestão, que é uma das suas principais bandeiras, foi questionado por alguns economistas. Como o senhor responde a essas críticas específicas?"
RESPOSTA: "Vamos lá, Teresa, a primeira etapa do nosso choque de gestão foi justamente... não vou entrar no mérito das críticas específicas, mas posso dizer que os resultados falam por si mesmos. Reduzimos o déficit em 40%..."
SAÍDA: {"objetivo_central": "Responder às críticas específicas dos economistas sobre o choque de gestão.", "mesmo_tema": true, "aborda_ponto_especifico": "Fala do choque de gestão (mesmo tema), mas recusa abordar as críticas específicas.", "rotulo": "PARCIAL"}

Responda APENAS com um objeto JSON válido contendo as chaves "objetivo_central", "mesmo_tema", "aborda_ponto_especifico" e "rotulo" (cujo valor deve ser estritamente "RESPONDE", "PARCIAL" ou "ESQUIVA")."""

# ==========================================
# 4. FUNÇÃO DE CHAMADA PARA A GROQ (QWEN)
# ==========================================
def classificar_qwen(pergunta: str, resposta: str, model_name=GROQ_MODEL) -> dict:
    user_msg = f'Analise o par abaixo:\nPERGUNTA: "{pergunta}"\nRESPOSTA: "{resposta}"'
    for tentativa in range(3):
        try:
            chat_completion = groq_client.chat.completions.create(
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg}
                ],
                model=model_name,
                temperature=0.0,
                response_format={"type": "json_object"},
            )
            return json.loads(chat_completion.choices[0].message.content)
        except Exception as e:
            print(f"Erro Groq (tentativa {tentativa+1}): {e}")
            time.sleep(5)
    return {"rotulo": "ERRO", "objetivo_central": "", "aborda_ponto_especifico": ""}

# ==========================================
# 5. CARREGAMENTO E EXECUÇÃO EM LOTE
# ==========================================
if os.path.exists("resultados_anotacao_llms.csv"):
    df = pd.read_csv("resultados_anotacao_llms.csv")
else:
    df = pd.read_excel("guia_anotacao (1).xlsx", sheet_name="Anotação")

for col in ["QWEN_ROTULO", "QWEN_JUSTIFICATIVA"]:
    if col not in df.columns:
        df[col] = None

print("\n🔍 Testando a linha 1 com o Qwen via Groq...")
teste_q = classificar_qwen(df.loc[1, "PERGUNTA"], df.loc[1, "RESPOSTA"])
print("Resultado Teste Qwen:", teste_q.get("rotulo"), "| Justificativa:", teste_q.get("aborda_ponto_especifico"))

if teste_q.get("rotulo") == "ERRO":
    raise RuntimeError("⚠️ O teste inicial falhou. Verifique a mensagem acima.")

# Loop principal pelas 694 linhas
for idx, row in tqdm(df.iterrows(), total=len(df)):
    pergunta = str(row["PERGUNTA"])
    resposta = str(row["RESPOSTA"])

    if pd.isna(df.at[idx, "QWEN_ROTULO"]) or df.at[idx, "QWEN_ROTULO"] == "ERRO":
        res_q = classificar_qwen(pergunta, resposta)
        df.at[idx, "QWEN_ROTULO"] = res_q.get("rotulo")
        df.at[idx, "QWEN_JUSTIFICATIVA"] = res_q.get("aborda_ponto_especifico")

    # Salva checkpoint a cada 20 linhas
    if idx % 20 == 0:
        df.to_csv("resultados_anotacao_llms.csv", index=False)

    time.sleep(1.0)

df.to_csv("resultados_anotacao_llms.csv", index=False)
print("🎉 Concluído! Arquivo salvo em resultados_anotacao_llms.csv com as classificações do Qwen.")

In [ ]:
import os
import json
import time
import pandas as pd
from tqdm import tqdm
from pydantic import BaseModel
from typing import Literal
from groq import Groq

# 1. Sua chave da Groq
# Credential removed for public archive. Use GROQ_API_KEY from environment or Colab Secrets.
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])
GROQ_MODEL = "qwen/qwen3.8-27b"

# 2. Schema de saída e o System Prompt alinhado ao guia
class AnotacaoResponsividade(BaseModel):
    objetivo_central: str
    mesmo_tema: bool
    aborda_ponto_especifico: str
    rotulo: Literal["RESPONDE", "PARCIAL", "ESQUIVA"]

SYSTEM_PROMPT = """Você é um anotador especialista em Ciência Política e Linguística Computacional analisando entrevistas políticas em português brasileiro do programa Roda Viva.
Sua tarefa é classificar o nível de responsividade da RESPOSTA do político em relação à PERGUNTA do jornalista em uma de três categorias: RESPONDE, PARCIAL ou ESQUIVA.

### DEFINIÇÕES DAS CATEGORIAS
- RESPONDE: O político aborda DIRETAMENTE o ponto central da pergunta. A resposta satisfaz o objetivo comunicativo principal do jornalista, mesmo que não cubra todos os aspectos secundários.
- PARCIAL: O político aborda o TEMA da pergunta, mas evita aspectos-chave, introduz digressões significativas ou dá uma resposta incompleta que deixa o ponto central sem resposta.
- ESQUIVA: O político redireciona para outro assunto, responde a uma pergunta diferente da feita, ou usa estratégias retóricas (atacar o entrevistador, mudar de tema) para evitar o ponto central.

### ÁRVORE DE DECISÃO (SIGA ESTES 3 PASSOS)
PASSO 1: Identifique o OBJETIVO CENTRAL da pergunta. Pergunte: "O que o jornalista quer saber de fato?" Ignore a conversa de abertura e contextualizações longas. Foque no que é pedido.
PASSO 2: O político fala sobre o MESMO TEMA da pergunta?
-> NÃO: classifique como ESQUIVA (o político mudou de assunto completamente).
-> SIM: vá para o Passo 3.
PASSO 3: O político aborda o PONTO ESPECÍFICO que foi perguntado?
-> SIM: classifique como RESPONDE (ele respondeu diretamente o que foi perguntado).
-> NÃO / PARCIALMENTE: classifique como PARCIAL (ele falou do tema, mas evitou o ponto central).

### REGRAS PRAGMÁTICAS CRÍTICAS (ATENÇÃO)
1. Extensão != qualidade: Uma resposta longa NÃO é automaticamente RESPONDE. Um político pode falar por vários minutos sobre o tema sem responder à pergunta específica. Foque no cumprimento do objetivo central, não no tamanho do texto. Da mesma forma, uma resposta curta e direta que atende ao que foi perguntado é RESPONDE.
2. Pergunta com duas ou mais partes: Foque na parte PRINCIPAL (geralmente a última, mais específica). Se o político responde apenas à parte secundária, classifique como PARCIAL.
3. Político elogia o entrevistador e desvia: Se o desvio é claro e o ponto central não é abordado -> ESQUIVA.
4. Resposta vaga ou genérica sobre o tema: Se não satisfaz o que foi perguntado especificamente -> PARCIAL, não RESPONDE.
5. Recusa explícita ou desconhecimento: Se o político diz diretamente "não sei" ou "não posso responder", isso É uma resposta direta -> RESPONDE.

Responda APENAS com um objeto JSON válido contendo as chaves "objetivo_central", "mesmo_tema", "aborda_ponto_especifico" e "rotulo" (cujo valor deve ser estritamente "RESPONDE", "PARCIAL" ou "ESQUIVA")."""

def classificar_qwen(pergunta: str, resposta: str) -> dict:
    user_msg = f'Analise o par abaixo:\nPERGUNTA: "{pergunta}"\nRESPOSTA: "{resposta}"'
    for tentativa in range(3):
        try:
            chat_completion = groq_client.chat.completions.create(
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg}
                ],
                model=GROQ_MODEL,
                temperature=0.0,
                response_format={"type": "json_object"},
            )
            return json.loads(chat_completion.choices[0].message.content)
        except Exception as e:
            print(f"Erro Groq (tentativa {tentativa+1}): {e}")
            time.sleep(10) # Espera 10 segundos se bater em limite de velocidade antes de tentar denovo
    return {"rotulo": "ERRO", "objetivo_central": "", "aborda_ponto_especifico": ""}

# 3. Carrega o arquivo que já tem o progresso salvo
if os.path.exists("resultados_anotacao_llms.csv"):
    df = pd.read_csv("resultados_anotacao_llms.csv")
else:
    raise FileNotFoundError("O arquivo resultados_anotacao_llms.csv não foi encontrado na pasta do Colab.")

# Mostra quantas linhas já estão prontas
concluidas = df["QWEN_ROTULO"].notna().sum() & (df["QWEN_ROTULO"] != "ERRO")
print(f"📊 Status atual: {concluidas} de {len(df)} linhas já estão anotadas e salvas.")

# 4. Continua o loop a partir de onde parou
for idx, row in tqdm(df.iterrows(), total=len(df)):
    # Pula as linhas que já foram preenchidas com sucesso
    if pd.notna(row["QWEN_ROTULO"]) and row["QWEN_ROTULO"] != "ERRO":
        continue

    pergunta = str(row["PERGUNTA"])
    resposta = str(row["RESPOSTA"])

    res_q = classificar_qwen(pergunta, resposta)
    df.at[idx, "QWEN_ROTULO"] = res_q.get("rotulo")
    df.at[idx, "QWEN_JUSTIFICATIVA"] = res_q.get("aborda_ponto_especifico")

    # Salva checkpoint a cada 20 linhas
    if idx % 20 == 0:
        df.to_csv("resultados_anotacao_llms.csv", index=False)

    time.sleep(1.5) # Pausa de 1.5s entre requisições para respeitar o limite de velocidade (RPM)

df.to_csv("resultados_anotacao_llms.csv", index=False)
print("🎉 Processamento concluído! Base 100% anotada salva em resultados_anotacao_llms.csv")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix

# 1. Carrega o arquivo final com as anotações do Qwen
df = pd.read_csv("resultados_anotacao_llms.csv")

print(f"Total de linhas na planilha: {len(df)}")
print("Distribuição dos rótulos gerados pelo Qwen:")
print(df["QWEN_ROTULO"].value_counts(dropna=False))

# 2. Compara com os dados humanos se as colunas existirem
if "ANOTACAO_HUMANA" in df.columns and "Anotacao 2" in df.columns and "Anotador 3" in df.columns:
    # Limpa e cria a maioria humana
    for col in ['ANOTACAO_HUMANA', 'Anotacao 2', 'Anotador 3', 'QWEN_ROTULO']:
        df[col] = df[col].astype(str).str.strip().str.upper()
        df.loc[df[col] == 'NAN', col] = np.nan

    def get_maj(row):
        vals = [row['ANOTACAO_HUMANA'], row['Anotacao 2'], row['Anotador 3']]
        vals_valid = [v for v in vals if pd.notna(v)]
        for v in set(vals_valid):
            if vals_valid.count(v) >= 2:
                return v
        return np.nan

    df['MAJ_HUMANA'] = df.apply(get_maj, axis=1)

    # Subconjunto com maioria humana válida e Qwen válido
    valido = df.dropna(subset=['MAJ_HUMANA', 'QWEN_ROTULO']).copy()
    valido = valido[valido['QWEN_ROTULO'] != 'ERRO']

    print(f"\nAmostra comparável com humanos: {len(valido)} linhas")

    # Kappa de Cohen
    kappa = cohen_kappa_score(valido['MAJ_HUMANA'], valido['QWEN_ROTULO'])
    print(f"\n📈 Cohen's Kappa (Qwen vs Maioria Humana): {kappa:.3f}")

    print("\n📊 Relatório de Classificação (Qwen vs Maioria Humana):")
    print(classification_report(valido['MAJ_HUMANA'], valido['QWEN_ROTULO'], zero_division=0))

    print("\n📉 Matriz de Confusão (Linhas = Humano, Colunas = Qwen):")
    print(confusion_matrix(valido['MAJ_HUMANA'], valido['QWEN_ROTULO'], labels=["RESPONDE", "PARCIAL", "ESQUIVA"]))
else:
    print("As colunas de anotação humana não foram encontradas no CSV. Verifique os nomes das colunas.")

In [ ]:
# 1. Instalar bibliotecas se necessário
!pip install -q groq pandas tqdm

import os
import json
import time
import pandas as pd
from tqdm import tqdm
from groq import Groq

# 2. Sua nova chave da Groq
# Credential removed for public archive. Use GROQ_API_KEY from environment or Colab Secrets.
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])
GROQ_MODEL = "qwen/qwen3.8-27b"

# 3. Prompt Simples (Zero-Shot) isolado — sem árvore de decisão e sem regras estritas
SYSTEM_PROMPT_SIMPLES = """Avalie se o político que respondeu à pergunta na entrevista abaixo respondeu aos pontos principais integralmente (RESPONDE), parcialmente (PARCIAL) ou não respondeu (ESQUIVA).
Responda APENAS com um objeto JSON válido contendo a chave "rotulo" com um dos valores exatos: "RESPONDE", "PARCIAL" ou "ESQUIVA"."""

def classificar_qwen_simples(pergunta: str, resposta: str) -> str:
    user_msg = f'PERGUNTA: "{pergunta}"\nRESPOSTA: "{resposta}"'
    for tentativa in range(3):
        try:
            chat_completion = groq_client.chat.completions.create(
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT_SIMPLES},
                    {"role": "user", "content": user_msg}
                ],
                model=GROQ_MODEL,
                temperature=0.0,
                response_format={"type": "json_object"},
            )
            dados = json.loads(chat_completion.choices[0].message.content)
            return str(dados.get("rotulo", "ERRO")).strip().upper()
        except Exception as e:
            print(f"Erro Groq Simples (tentativa {tentativa+1}): {e}")
            time.sleep(8)
    return "ERRO"

# 4. Carrega o CSV existente
if os.path.exists("resultados_anotacao_llms.csv"):
    df = pd.read_csv("resultados_anotacao_llms.csv")
else:
    raise FileNotFoundError("O arquivo resultados_anotacao_llms.csv não foi encontrado na pasta do Colab.")

if "QWEN_SIMPLES_ROTULO" not in df.columns:
    df["QWEN_SIMPLES_ROTULO"] = None

# Seleciona as linhas que já têm a anotação válida do Qwen com guia para fazermos o pareamento exato
linhas_alvo = df[df["QWEN_ROTULO"].isin(["RESPONDE", "PARCIAL", "ESQUIVA"])].index
print(f"🚀 Iniciando anotação com Prompt Simples nas {len(linhas_alvo)} linhas pareadas...")

for idx in tqdm(linhas_alvo):
    if pd.notna(df.at[idx, "QWEN_SIMPLES_ROTULO"]) and df.at[idx, "QWEN_SIMPLES_ROTULO"] != "ERRO":
        continue
    df.at[idx, "QWEN_SIMPLES_ROTULO"] = classificar_qwen_simples(
        str(df.at[idx, "PERGUNTA"]),
        str(df.at[idx, "RESPOSTA"])
    )
    if idx % 25 == 0:
        df.to_csv("resultados_anotacao_llms.csv", index=False)
    time.sleep(1.0)

df.to_csv("resultados_anotacao_llms.csv", index=False)
print("🎉 Concluído! Coluna QWEN_SIMPLES_ROTULO gerada e salva com sucesso.")

In [ ]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score, classification_report, confusion_matrix

df = pd.read_csv("resultados_anotacao_llms.csv")

# Prepara a maioria humana e limpa os rótulos
for col in ['ANOTACAO_HUMANA', 'Anotacao 2', 'Anotador 3', 'QWEN_ROTULO', 'QWEN_SIMPLES_ROTULO']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.upper()
        df.loc[df[col] == 'NAN', col] = None

def get_maj(row):
    vals = [row['ANOTACAO_HUMANA'], row['Anotacao 2'], row['Anotador 3']]
    vals_valid = [v for v in vals if pd.notna(v)]
    for v in set(vals_valid):
        if vals_valid.count(v) >= 2:
            return v
    return None

df['MAJ_HUMANA'] = df.apply(get_maj, axis=1)

# Filtra a amostra estritamente pareada (248 linhas)
validos = df.dropna(subset=['MAJ_HUMANA', 'QWEN_ROTULO', 'QWEN_SIMPLES_ROTULO']).copy()
validos = validos[(validos['QWEN_ROTULO'] != 'ERRO') & (validos['QWEN_SIMPLES_ROTULO'] != 'ERRO')]

print(f"📊 Amostra pareada controlada: {len(validos)} linhas\n")

print("="*60)
print("--- 1. QWEN COM PROMPT SIMPLES (ZERO-SHOT) vs. MAIORIA HUMANA ---")
print("="*60)
print(f"Kappa de Cohen: {cohen_kappa_score(validos['MAJ_HUMANA'], validos['QWEN_SIMPLES_ROTULO']):.3f}")
print("\nRelatório de Classificação:")
print(classification_report(validos['MAJ_HUMANA'], validos['QWEN_SIMPLES_ROTULO'], digits=3, zero_division=0))

print("\n" + "="*60)
print("--- 2. QWEN COM GUIA PRAGMÁTICO (CODEBOOK-ALIGNED) vs. MAIORIA HUMANA ---")
print("="*60)
print(f"Kappa de Cohen: {cohen_kappa_score(validos['MAJ_HUMANA'], validos['QWEN_ROTULO']):.3f}")
print("\nRelatório de Classificação:")
print(classification_report(validos['MAJ_HUMANA'], validos['QWEN_ROTULO'], digits=3, zero_division=0))

print("\n" + "="*60)
print("--- 3. MATRIZ DE CONFUSÃO (QWEN SIMPLES) ---")
print("="*60)
print(confusion_matrix(validos['MAJ_HUMANA'], validos['QWEN_SIMPLES_ROTULO'], labels=["RESPONDE", "PARCIAL", "ESQUIVA"]))

print("\n" + "="*60)
print("--- 4. MATRIZ DE CONFUSÃO (QWEN COM GUIA PRAGMÁTICO) ---")
print("="*60)
print(confusion_matrix(validos['MAJ_HUMANA'], validos['QWEN_ROTULO'], labels=["RESPONDE", "PARCIAL", "ESQUIVA"]))